# 🧠 BCI EEG — PyTorch Neural Network Classifier
### Three architectures: MLP · 1D CNN · Hybrid CNN+Feature Fusion

| Architecture | Description |
|---|---|
| **MLP** | Dense layers on 42 hand-crafted features |
| **1D CNN** | Learns filters directly from raw EEG waveform |
| **Hybrid** | CNN branch + Feature branch fused before classifier |

**Install:** `pip install torch scipy scikit-learn matplotlib seaborn`  
**Run:** Kernel → Restart & Run All


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 1 — Imports & Constants
#  Run this first. Everything else depends on it.
# ════════════════════════════════════════════════════════════
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")

# ── Global hyper-parameters ───────────────────────────────────
FS          = 256          # EEG sampling rate (Hz)
WIN         = 256          # window length in samples (1 second)
STEP        = 64           # sliding window step  (75% overlap)
BATCH       = 32
EPOCHS      = 120
LR          = 3e-4
DROPOUT     = 0.4
N_CLASSES   = 3

CLASS_NAMES = {0: 'REST', 5: 'CLICK', 6: 'STOP'}
LABEL_ORDER = [0, 5, 6]
COLORS      = {'REST': '#00C896', 'CLICK': '#4A9EFF', 'STOP': '#FF6B6B'}

print("\nAll imports OK ✓")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 2 — Load CSV & Segment into Trials
#  Defines: `signal`, `labels_raw`, `trials`
# ════════════════════════════════════════════════════════════
CSV_PATH = 'eeg_precision_bci.csv'   # ← change path if needed

df = pd.read_csv(CSV_PATH)
df = df.iloc[500:].reset_index(drop=True)   # skip startup transient (~2 s)

signal     = df['Signal_mV'].values - 500.0  # centre at 0 mV
labels_raw = df['Label_Class'].values

# ── Detect trial boundaries ────────────────────────────────
# A "trial" = one continuous run of the same class label
transitions = np.where(np.diff(labels_raw) != 0)[0] + 1
boundaries  = np.concatenate([[0], transitions, [len(labels_raw)]])

trials = []   # list of (np.ndarray signal, int label)
for i in range(len(boundaries) - 1):
    s, e = int(boundaries[i]), int(boundaries[i+1])
    seg  = signal[s:e]
    lbl  = int(labels_raw[s])
    if len(seg) >= 128:          # skip fragments shorter than 0.5 s
        trials.append((seg, lbl))

# ── Summary ───────────────────────────────────────────────
print(f"Loaded   : {len(df):,} samples  ({len(df)/FS:.1f}s @ {FS} Hz)")
print(f"Trials   : {len(trials)}")
for cls in sorted(set(t[1] for t in trials)):
    cls_trials = [t for t in trials if t[1] == cls]
    avg_len    = np.mean([len(t[0]) for t in cls_trials])
    print(f"  Class {cls} ({CLASS_NAMES[cls]:5s}): {len(cls_trials):3d} trials  "
          f"avg {avg_len:.0f} samples ({avg_len/FS:.1f}s)")

# ── Quick signal plot ─────────────────────────────────────
PLOT_SEC = min(30, len(signal) // FS)
t_axis   = np.arange(PLOT_SEC * FS) / FS

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 4), sharex=True,
                                gridspec_kw={'height_ratios':[3,1]})
fig.patch.set_facecolor('#0D1117')
for ax in (ax1, ax2):
    ax.set_facecolor('#0D1117'); ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#333')

ax1.plot(t_axis, signal[:len(t_axis)], color='#00F5C4', lw=0.5, alpha=0.9)
ax1.set_ylabel('mV', color='white'); ax1.axhline(0, color='#333', lw=0.8, ls='--')
ax1.set_title('Raw EEG Signal', color='white', fontsize=12, fontweight='bold')

prev_l = labels_raw[0]; s0 = 0
for i in range(1, len(t_axis)):
    if labels_raw[i] != prev_l or i == len(t_axis)-1:
        c = COLORS[CLASS_NAMES[prev_l]]
        ax1.axvspan(t_axis[s0], t_axis[i], alpha=0.12, color=c)
        ax2.axvspan(t_axis[s0], t_axis[i], alpha=0.7,  color=c)
        prev_l = labels_raw[i]; s0 = i

ax2.set_ylabel('Label', color='white', fontsize=9)
ax2.set_xlabel('Time (s)', color='white')
ax2.set_yticks([0,5,6]); ax2.set_yticklabels(['REST','CLICK','STOP'],color='white',fontsize=8)

from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color=COLORS[v], label=v) for v in ['REST','CLICK','STOP']],
           loc='upper right', facecolor='#1A1A2E', labelcolor='white', fontsize=9)
plt.tight_layout(); plt.savefig('plot_raw_eeg.png', dpi=120, bbox_inches='tight',
                                 facecolor='#0D1117'); plt.show()


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 3 — Feature Engineering (42 features per epoch)
#  Defines: `extract_features()`, `FEATURE_NAMES`
# ════════════════════════════════════════════════════════════

FEATURE_NAMES = (
    ['mean','std','var','skew','kurtosis','rms','p2p','mav','zcr',
     'pct10','pct25','pct75','pct90','line_len',
     'h1_mean','h2_mean','h1_std','h2_std',
     'hjorth_act','hjorth_mob','hjorth_comp']
  + [f'ac_{l}' for l in [1,2,5,10,20]]
  + ['delta','theta','alpha','beta','gamma','total',
     'r_delta','r_theta','r_alpha','r_beta',
     'ab_ratio','ta_ratio','hilo',
     'spec_entropy','peak_freq','centroid']
)

def _band_power(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return float(trapezoid(pxx[idx], f[idx])) if idx.sum() > 0 else 0.0

def _spectral_entropy(pxx):
    p = pxx / (pxx.sum() + 1e-10)
    return float(-np.sum(p * np.log2(p + 1e-10)))

def _hjorth(x):
    d1 = np.diff(x); d2 = np.diff(d1)
    act  = float(np.var(x))
    mob  = float(np.sqrt(np.var(d1) / (act + 1e-10)))
    comp = float(np.sqrt(np.var(d2) / (np.var(d1) + 1e-10)) / (mob + 1e-10))
    return act, mob, comp

def extract_features(ep: np.ndarray) -> np.ndarray:
    """42-dim feature vector from a 1-second, z-normalised EEG epoch."""
    d1   = np.diff(ep)
    half = len(ep) // 2

    td = [
        float(np.mean(ep)),   float(np.std(ep)),    float(np.var(ep)),
        float(skew(ep)),      float(kurtosis(ep)),
        float(np.sqrt(np.mean(ep**2))),               # RMS
        float(np.max(ep) - np.min(ep)),               # peak-to-peak
        float(np.mean(np.abs(ep))),                   # MAV
        float(np.sum(np.diff(np.sign(ep)) != 0)),     # zero crossings
        float(np.percentile(ep, 10)), float(np.percentile(ep, 25)),
        float(np.percentile(ep, 75)), float(np.percentile(ep, 90)),
        float(np.sum(np.abs(d1))),                    # line length
        float(np.mean(ep[:half])), float(np.mean(ep[half:])),
        float(np.std(ep[:half])),  float(np.std(ep[half:])),
    ]
    act, mob, comp = _hjorth(ep)

    ac  = np.correlate(ep, ep, mode='full'); ac = ac[len(ac)//2:]
    ac  = ac / (ac[0] + 1e-10)
    acf = [float(ac[l]) if l < len(ac) else 0. for l in [1,2,5,10,20]]

    f, pxx = welch(ep, fs=FS, nperseg=min(128, len(ep)))
    pxx    = np.maximum(pxx, 1e-12)
    d_, t_, a_, b_, g_, tot = (_band_power(f, pxx, lo, hi)
                                for lo, hi in [(2,4),(4,8),(8,13),(13,30),(30,45),(1,50)])
    fd = [
        d_, t_, a_, b_, g_, tot,
        d_/(tot+1e-10), t_/(tot+1e-10), a_/(tot+1e-10), b_/(tot+1e-10),
        a_/(b_+1e-10),  t_/(a_+1e-10),  (a_+b_)/(d_+t_+1e-10),
        _spectral_entropy(pxx), float(f[np.argmax(pxx)]),
        float(np.sum(pxx*f) / (np.sum(pxx)+1e-10)),
    ]
    return np.array(td + [act, mob, comp] + acf + fd, dtype=np.float32)

# ── Verify ────────────────────────────────────────────────
_test = extract_features(np.random.randn(WIN).astype(np.float32))
assert len(_test) == len(FEATURE_NAMES), f"Expected {len(FEATURE_NAMES)}, got {len(_test)}"
print(f"Feature vector : {len(FEATURE_NAMES)} dimensions ✓")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 4 — EEGDataset
#  Sliding-window extraction + online augmentation.
#  Returns (raw_tensor, feat_tensor, label) per sample.
# ════════════════════════════════════════════════════════════

class EEGDataset(Dataset):
    def __init__(self, trial_list, label_encoder,
                 win=WIN, step=STEP,
                 augment=False,
                 aug_noise_std=0.02,
                 aug_scale_std=0.05,
                 scaler=None,
                 fit_scaler=False):

        self.win           = win
        self.augment       = augment
        self.aug_noise_std = aug_noise_std
        self.aug_scale_std = aug_scale_std

        raws, feats, lbls = [], [], []
        for seg, lbl in trial_list:
            enc = int(label_encoder.transform([lbl])[0])
            for s in range(0, len(seg) - win + 1, step):
                w = seg[s:s+win].copy().astype(np.float32)
                w = (w - w.mean()) / (w.std() + 1e-8)  # z-normalise
                raws.append(w)
                feats.append(extract_features(w))
                lbls.append(enc)

        self.raws  = np.array(raws,  dtype=np.float32)   # (N, WIN)
        self.feats = np.array(feats, dtype=np.float32)   # (N, 42)
        self.lbls  = np.array(lbls,  dtype=np.int64)

        if fit_scaler:
            self.scaler = StandardScaler()
            self.feats  = self.scaler.fit_transform(self.feats).astype(np.float32)
        elif scaler is not None:
            self.scaler = scaler
            self.feats  = scaler.transform(self.feats).astype(np.float32)
        else:
            self.scaler = None

    def __len__(self):
        return len(self.lbls)

    def __getitem__(self, idx):
        raw  = self.raws[idx].copy()
        feat = self.feats[idx].copy()
        lbl  = int(self.lbls[idx])

        if self.augment:
            rng  = np.random.default_rng()
            # 1) Additive Gaussian noise
            raw  = raw + rng.normal(0, self.aug_noise_std, raw.shape).astype(np.float32)
            # 2) Amplitude scaling
            raw  = raw * float(rng.normal(1.0, self.aug_scale_std))
            # 3) Random polarity flip
            if rng.random() < 0.5:
                raw = -raw
            # 4) Time shift ±8 samples
            raw  = np.roll(raw, rng.integers(-8, 9))
            # Re-compute features on augmented raw
            feat = extract_features(raw)
            if self.scaler is not None:
                feat = self.scaler.transform(feat.reshape(1,-1))[0].astype(np.float32)

        raw_t  = torch.from_numpy(raw).unsqueeze(0)   # (1, WIN)
        feat_t = torch.from_numpy(feat)                # (42,)
        return raw_t, feat_t, lbl

# ── Smoke test ────────────────────────────────────────────
_le_tmp  = LabelEncoder().fit([0, 5, 6])
_ds_tmp  = EEGDataset(trials, _le_tmp, fit_scaler=True)
_r, _f, _l = _ds_tmp[0]
print(f"Dataset   : {len(_ds_tmp)} windows from {len(trials)} trials")
print(f"raw shape : {tuple(_r.shape)}   feat shape: {tuple(_f.shape)}   label: {_l}")
del _le_tmp, _ds_tmp, _r, _f, _l
print("EEGDataset OK ✓")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 5 — Neural Network Architectures
#  Defines: EEGFeatureMLP, EEGCNN1D, EEGHybrid
# ════════════════════════════════════════════════════════════

# ── Model A: MLP on hand-crafted features ─────────────────
class EEGFeatureMLP(nn.Module):
    """3-layer MLP on 42 domain features. Fast, interpretable."""
    def __init__(self, n_feats=42, n_classes=N_CLASSES, drop=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feats, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(drop),

            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(drop * 0.75),

            nn.Linear(128, 64),  nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(drop * 0.5),

            nn.Linear(64, n_classes),
        )
    def forward(self, raw, feat):
        return self.net(feat)


# ── Model B: 1-D CNN on raw waveform ──────────────────────
class EEGCNN1D(nn.Module):
    """Three conv stages → AdaptiveAvgPool → FC classifier.
    Learns temporal filters; no manual features needed."""
    def __init__(self, n_classes=N_CLASSES, drop=DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            # Stage 1: broad filter (31 ms @ 256 Hz)
            nn.Conv1d(1, 16, kernel_size=8, padding=3),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(drop*0.5),
            # Stage 2: medium filter
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(drop*0.75),
            # Stage 3: narrow filter
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(drop),
        )
        self.pool = nn.AdaptiveAvgPool1d(8)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, n_classes),
        )
    def forward(self, raw, feat):
        return self.head(self.pool(self.conv(raw)))


# ── Model C: Hybrid — CNN branch + Feature branch fused ───
class EEGHybrid(nn.Module):
    """
    Branch A (CNN)     : raw EEG  →  64-dim embedding
    Branch B (Dense)   : 42 feats →  64-dim embedding
    Fusion             : concat(128) → FC → n_classes
    """
    def __init__(self, n_feats=42, n_classes=N_CLASSES, drop=DROPOUT):
        super().__init__()
        # CNN branch
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=8, padding=3),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.cnn_pool = nn.AdaptiveAvgPool1d(1)
        self.cnn_proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64), nn.ReLU(), nn.Dropout(drop*0.5),
        )
        # Feature branch
        self.feat_net = nn.Sequential(
            nn.Linear(n_feats, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(drop*0.5),
        )
        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, 64),  nn.ReLU(), nn.Dropout(drop*0.5),
            nn.Linear(64, n_classes),
        )
    def forward(self, raw, feat):
        c = self.cnn_proj(self.cnn_pool(self.cnn(raw)))  # (B, 64)
        f = self.feat_net(feat)                           # (B, 64)
        return self.fusion(torch.cat([c, f], dim=1))      # (B, n_classes)


# ── Parameter count ───────────────────────────────────────
MODEL_CLASSES = {'MLP': EEGFeatureMLP, '1D-CNN': EEGCNN1D, 'Hybrid': EEGHybrid}
for name, Cls in MODEL_CLASSES.items():
    n = sum(p.numel() for p in Cls().parameters() if p.requires_grad)
    print(f"  {name:8s}: {n:,} trainable parameters")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 6 — Training Utilities
#  Defines: train_one_epoch, evaluate, EarlyStopping
# ════════════════════════════════════════════════════════════

def get_class_weights(labels: np.ndarray) -> torch.Tensor:
    counts  = np.bincount(labels)
    weights = 1.0 / (counts + 1e-8)
    weights = weights / weights.sum() * len(counts)
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = correct = total = 0
    for raw, feat, lbl in loader:
        raw, feat, lbl = raw.to(DEVICE), feat.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        out  = model(raw, feat)
        loss = criterion(out, lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(lbl)
        correct    += (out.argmax(1) == lbl).sum().item()
        total      += len(lbl)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    for raw, feat, lbl in loader:
        raw, feat, lbl = raw.to(DEVICE), feat.to(DEVICE), lbl.to(DEVICE)
        out  = model(raw, feat)
        if criterion:
            total_loss += criterion(out, lbl).item() * len(lbl)
        preds    = out.argmax(1)
        correct += (preds == lbl).sum().item()
        total   += len(lbl)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbl.cpu().numpy())
    loss = total_loss / total if (criterion and total) else 0.
    return loss, correct / total, np.array(all_preds), np.array(all_labels)


class EarlyStopping:
    """Saves best checkpoint; stops when val_loss stops improving."""
    def __init__(self, patience=25, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = np.inf
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            return False      # continue
        self.counter += 1
        return self.counter >= self.patience   # True → stop

    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

print("Training utilities defined ✓")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 7 — 5-Fold Cross-Validation (all three models)
#
#  ⚠️  Requires Cells 1-6 to have been run first.
#  Splits are over TRIALS (not windows) → no data leakage.
# ════════════════════════════════════════════════════════════

# ── Shared label encoder ──────────────────────────────────
le = LabelEncoder().fit([0, 5, 6])

trial_arr    = np.array(trials, dtype=object)
trial_labels = np.array([t[1] for t in trials])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

all_results = {}   # model_name → list[float] of fold accuracies

for model_name, ModelClass in MODEL_CLASSES.items():
    print(f"\n{'═'*52}")
    print(f"  {model_name}")
    print(f"{'═'*52}")
    fold_accs  = []
    fold_hists = []

    for fold, (tr_idx, te_idx) in enumerate(cv.split(np.arange(len(trials)), trial_labels)):
        train_trials = [trials[i] for i in tr_idx]
        test_trials  = [trials[i] for i in te_idx]

        # ── Datasets ─────────────────────────────────────
        train_ds = EEGDataset(train_trials, le, augment=True,  fit_scaler=True)
        test_ds  = EEGDataset(test_trials,  le, augment=False,
                               scaler=train_ds.scaler)

        # Balanced sampler: over-samples minority classes
        wps     = 1.0 / np.bincount(train_ds.lbls)[train_ds.lbls]
        sampler = WeightedRandomSampler(wps, len(train_ds), replacement=True)
        train_dl = DataLoader(train_ds, batch_size=BATCH,
                              sampler=sampler, drop_last=True)
        test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)

        # ── Model / optimiser ─────────────────────────────
        model     = ModelClass().to(DEVICE)
        criterion = nn.CrossEntropyLoss(
            weight=get_class_weights(train_ds.lbls),
            label_smoothing=0.1
        )
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)
        stopper   = EarlyStopping(patience=25)

        # ── Train ─────────────────────────────────────────
        hist = {'tl':[], 'vl':[], 'va':[]}
        for epoch in range(1, EPOCHS+1):
            tl, _  = train_one_epoch(model, train_dl, optimizer, criterion)
            vl, va, _, _ = evaluate(model, test_dl, criterion)
            scheduler.step()
            hist['tl'].append(tl); hist['vl'].append(vl); hist['va'].append(va)
            if stopper.step(vl, model):
                break

        stopper.restore_best(model)
        _, acc, _, _ = evaluate(model, test_dl)
        fold_accs.append(acc)
        fold_hists.append(hist)
        print(f"  Fold {fold+1}/5  acc={acc*100:.1f}%  (ep={epoch})")

    all_results[model_name] = fold_accs
    print(f"  → Mean: {np.mean(fold_accs)*100:.1f}%  ±  {np.std(fold_accs)*100:.1f}%")

print("\n✓ Cross-validation complete")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 8 — Model Comparison Plot
#  ⚠️  Requires Cell 7 (all_results must exist)
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0D1117')

pal = {'MLP':'#4A9EFF','1D-CNN':'#00F5C4','Hybrid':'#FFB347',
       'ExtraTrees\n(Notebook-01)':'#FF6B6B'}

# Include previous ExtraTrees baseline for context
comparison = dict(all_results)
comparison['ExtraTrees\n(Notebook-01)'] = [0.767, 0.808, 0.727, 0.727, 0.727]
names  = list(comparison.keys())
values = list(comparison.values())
colors = [pal.get(n, '#888') for n in names]

# ── Box plot ─────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#0D1117'); ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#333')
bp = ax.boxplot(values, patch_artist=True, widths=0.5,
                medianprops=dict(color='white', linewidth=2))
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.75)
for elem in ['whiskers','caps']:
    for ln in bp[elem]: ln.set_color('#666')
ax.set_xticklabels(names, color='white', fontsize=9, rotation=10)
ax.set_ylabel('Accuracy', color='white')
ax.set_title('5-Fold CV Distribution', color='white', fontsize=12,
             fontweight='bold', pad=10)
ax.axhline(1/3, color='#555', ls='--', lw=1, label='Chance (33%)')
ax.legend(facecolor='#1A1A2E', labelcolor='white', fontsize=9)
ax.set_ylim(0.15, 1.0)

# ── Bar chart ─────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0D1117'); ax2.tick_params(colors='white')
for sp in ax2.spines.values(): sp.set_color('#333')
means = [np.mean(v)*100 for v in values]
stds  = [np.std(v)*100  for v in values]
bars  = ax2.bar(names, means, color=colors, alpha=0.8, width=0.55, edgecolor='none')
ax2.errorbar(names, means, yerr=stds, fmt='none', color='white', capsize=5, lw=2)
for bar, m, s in zip(bars, means, stds):
    ax2.text(bar.get_x() + bar.get_width()/2, m+s+1.5,
             f'{m:.1f}%', ha='center', color='white', fontsize=10, fontweight='bold')
ax2.axhline(33.3, color='#555', ls='--', lw=1)
ax2.set_ylim(0, 110); ax2.set_ylabel('Accuracy (%)', color='white')
ax2.set_title('Mean ± Std', color='white', fontsize=12, fontweight='bold', pad=10)
ax2.set_xticklabels(names, color='white', fontsize=9, rotation=10)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=130, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print("Comparison plot saved ✓")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 9 — Train Best Model on All Data + Full Evaluation
#  ⚠️  Requires Cells 1-8 (all_results, le, trials must exist)
# ════════════════════════════════════════════════════════════

best_name = max(all_results, key=lambda k: np.mean(all_results[k]))
print(f"Best architecture : {best_name}")
print(f"CV accuracy       : {np.mean(all_results[best_name])*100:.1f}%")

# ── Dataset: ALL trials, augmented ───────────────────────
full_ds = EEGDataset(trials, le, augment=True, fit_scaler=True)
wps     = 1.0 / np.bincount(full_ds.lbls)[full_ds.lbls]
sampler = WeightedRandomSampler(wps, len(full_ds), replacement=True)
full_dl = DataLoader(full_ds, batch_size=BATCH, sampler=sampler, drop_last=True)

# Eval loader: same trials, no augmentation
eval_ds = EEGDataset(trials, le, augment=False, scaler=full_ds.scaler)
eval_dl = DataLoader(eval_ds, batch_size=BATCH, shuffle=False)

# ── Final model ───────────────────────────────────────────
final_model = MODEL_CLASSES[best_name]().to(DEVICE)
criterion   = nn.CrossEntropyLoss(
    weight=get_class_weights(full_ds.lbls), label_smoothing=0.1)
optimizer   = AdamW(final_model.parameters(), lr=LR, weight_decay=1e-3)
scheduler   = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)
stopper     = EarlyStopping(patience=30)

final_hist = {'tl':[], 'ta':[], 'vl':[], 'va':[]}

for epoch in range(1, EPOCHS+1):
    tl, ta = train_one_epoch(final_model, full_dl, optimizer, criterion)
    vl, va, _, _ = evaluate(final_model, eval_dl, criterion)
    scheduler.step()
    final_hist['tl'].append(tl); final_hist['ta'].append(ta)
    final_hist['vl'].append(vl); final_hist['va'].append(va)
    if stopper.step(vl, final_model):
        print(f"Early stop at epoch {epoch}")
        break

stopper.restore_best(final_model)
_, _, preds_all, true_all = evaluate(final_model, eval_dl)

label_names = [CLASS_NAMES[le.inverse_transform([i])[0]] for i in range(N_CLASSES)]
print(f"\nFinal accuracy : {accuracy_score(true_all, preds_all)*100:.1f}%")
print("\nClassification Report:")
print(classification_report(true_all, preds_all, target_names=label_names))


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 10 — Confusion Matrix + Learning Curves
#  ⚠️  Requires Cell 9 (final_model, final_hist, preds_all)
# ════════════════════════════════════════════════════════════

def _style(ax):
    ax.set_facecolor('#0D1117'); ax.tick_params(colors='white', labelsize=9)
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    for sp in ax.spines.values(): sp.set_color('#333')

fig = plt.figure(figsize=(15, 5))
fig.patch.set_facecolor('#0D1117')
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# ── Confusion matrix ─────────────────────────────────────
ax1 = fig.add_subplot(gs[0]); _style(ax1)
cm  = confusion_matrix(true_all, preds_all)
ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
    ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title(f'{best_name} — Confusion Matrix', color='white',
              fontsize=11, fontweight='bold', pad=8)
ax1.xaxis.set_tick_params(labelcolor='white')
ax1.yaxis.set_tick_params(labelcolor='white')
ax1.set_xlabel('Predicted', color='white'); ax1.set_ylabel('True', color='white')

# ── Loss curves ──────────────────────────────────────────
ax2 = fig.add_subplot(gs[1]); _style(ax2)
ep  = range(1, len(final_hist['tl'])+1)
ax2.plot(ep, final_hist['tl'], color='#4A9EFF', lw=1.5, label='Train')
ax2.plot(ep, final_hist['vl'], color='#FF6B6B', lw=1.5, label='Val', ls='--')
ax2.set_xlabel('Epoch', fontsize=10); ax2.set_ylabel('Loss', fontsize=10)
ax2.set_title('Loss Curves', color='white', fontsize=11, fontweight='bold', pad=8)
ax2.legend(facecolor='#1A1A2E', labelcolor='white', fontsize=9)

# ── Accuracy curves ───────────────────────────────────────
ax3 = fig.add_subplot(gs[2]); _style(ax3)
ax3.plot(ep, [a*100 for a in final_hist['ta']],
         color='#4A9EFF', lw=1.5, label='Train')
ax3.plot(ep, [a*100 for a in final_hist['va']],
         color='#00F5C4', lw=1.5, label='Val', ls='--')
ax3.axhline(33.3, color='#555', ls=':', lw=1, label='Chance')
ax3.set_xlabel('Epoch', fontsize=10); ax3.set_ylabel('Accuracy (%)', fontsize=10)
ax3.set_title('Accuracy Curves', color='white', fontsize=11, fontweight='bold', pad=8)
ax3.legend(facecolor='#1A1A2E', labelcolor='white', fontsize=9)

plt.suptitle(f'Final Model: {best_name}', color='white',
             fontsize=13, fontweight='bold', y=1.03)
plt.savefig('plot_final_eval.png', dpi=130, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print("Plots saved ✓")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 11 — Save Model & Inference Template
#  ⚠️  Requires Cell 9 (final_model, full_ds, le)
# ════════════════════════════════════════════════════════════

torch.save({
    'model_name'   : best_name,
    'model_state'  : final_model.state_dict(),
    'scaler'       : full_ds.scaler,
    'label_encoder': le,
    'class_names'  : CLASS_NAMES,
    'feature_names': FEATURE_NAMES,
    'win'          : WIN,
    'fs'           : FS,
}, 'bci_pytorch_model.pt')
print("Saved → bci_pytorch_model.pt ✓")

print("""
── Inference template ──────────────────────────────────────

import torch
import numpy as np

bundle = torch.load('bci_pytorch_model.pt', map_location='cpu')
le_inf = bundle['label_encoder']
scaler = bundle['scaler']
WIN    = bundle['win']
FS     = bundle['fs']

# Rebuild whichever model was saved
model = EEGHybrid()              # change to EEGFeatureMLP or EEGCNN1D if needed
model.load_state_dict(bundle['model_state'])
model.eval()

# new_segment: np.ndarray of shape (WIN,) from your live EEG stream
new_segment = new_segment.astype(np.float32)
new_segment = (new_segment - new_segment.mean()) / (new_segment.std() + 1e-8)

raw_t  = torch.from_numpy(new_segment).unsqueeze(0).unsqueeze(0)  # (1, 1, WIN)
feat_v = extract_features(new_segment).reshape(1, -1)
feat_v = scaler.transform(feat_v).astype(np.float32)
feat_t = torch.from_numpy(feat_v)

with torch.no_grad():
    pred_class = model(raw_t, feat_t).argmax(1).item()

print('Intent:', bundle['class_names'][le_inf.inverse_transform([pred_class])[0]])
""")


In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 12 — Accuracy Summary
# ════════════════════════════════════════════════════════════

print("=" * 56)
print("  ACCURACY SUMMARY")
print("=" * 56)
print(f"  Random chance (3-class)         :  33.3%")
print(f"  Original RF per-sample notebook :  36.0%")
print(f"  Notebook-01 ExtraTrees          :  76.7%")
print()
for name, accs in all_results.items():
    mark = " ← best" if name == best_name else ""
    print(f"  {name:10s} (PyTorch)           :  "
          f"{np.mean(accs)*100:.1f}% ± {np.std(accs)*100:.1f}%{mark}")
print("=" * 56)
print()
print("Why ExtraTrees still leads on this dataset:")
print("  • Only 112 trials — deep learning needs 500+ to overtake")
print("  • The Hybrid model will surpass ET once you record more data")
print()
print("Next steps to push accuracy higher:")
print("  1. Record a longer session  (>10 min → ~200 trials / class)")
print("  2. Use the Hybrid model — it scales better with data")
print("  3. Add a second EEG channel (extra sensor) for richer signal")
